In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib
import matplotlib.pyplot as plt
import random
import seaborn as sns
import pandas as pd
import scipy.stats as stats

import os
import numpy as np
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
from quad_model import *
from tensorflow.keras.models import Model, load_model
from  tqdm import tqdm

#from interpretable_splicing.models import NucleotideStructureTunerCNN
import mukund_utils as utils

random.seed(100)
np.random.seed(100)


def softplus(x):
    return np.log(1 + np.exp(x))

    
def get_random_model_input(num_samples=1, length=70):

    all_nucleotides = []
    all_seq = []
    all_struct = []
    all_wobble = []
    for i in range(num_samples):
        nucleotides = ''.join([random.choice('ACGU') for _ in range(length)])

        seq_oh, struct_oh, wobbles, structs, mfe = utils.create_input_data(seq_nts=[nucleotides], return_mfe=True)
        all_nucleotides.append(nucleotides)
        all_seq.append(seq_oh.squeeze())
        all_struct.append(struct_oh.squeeze())
        all_wobble.append(wobbles.squeeze())

    return all_nucleotides, np.stack(all_seq, axis=0), np.stack(all_struct, axis=0), np.stack(all_wobble, axis=0)


#@torch.no_grad()
def get_latent_embedding(rna_string: str, model):
    nucleotides = utils.add_flanking(rna_string, flanking_len=0)
    seq_oh, struct_oh, wobbles, structs, mfe = utils.create_input_data(seq_nts=[nucleotides], return_mfe=True)

#    if isinstance(model, torch.nn.Module):
#        nucleotide_input = torch.Tensor(seq_oh)
#        nucleotide_input = nucleotide_input.squeeze().unsqueeze(0)
#
#        activations_inclusion = model.activation(
#            model.nucleotide_filters['inclusion'](nucleotide_input.permute(0, 2, 1))
#        )#.sum(dim=-2)
#        activations_exclusion = model.activation(
#            model.nucleotide_filters['exclusion'](nucleotide_input.permute(0, 2, 1))
#        )#.sum(dim=-1)
#        # print(activations_inclusion.shape)
#        embeddings = torch.cat((activations_inclusion, activations_exclusion), dim=-2).numpy()
#    else: #keras
    qc_incl = model.get_layer('qc_incl')
    qc_skip = model.get_layer('qc_skip')
    print(seq_oh.shape)
    activations_inclusion = softplus(tf.squeeze(qc_incl(seq_oh), axis=0).numpy())
    activations_exclusion= softplus(tf.squeeze(qc_skip(seq_oh), axis=0).numpy())
    print(activations_inclusion.shape)
    #the ordering here of filters is intentional to make the graphics consistent across both the torch and tensorflow models
    embeddings = np.concatenate((activations_exclusion, activations_inclusion), axis=1).transpose()
    embeddings = np.expand_dims(embeddings, axis=0)

    return embeddings, structs, mfe[0]/len(rna_string)




In [ ]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
#model = NucleotideStructureTunerCNN.load_from_checkpoint(checkpoint_path='/home/andriy/PycharmProjects/interpretable_splicing/interpretable_splicing/model/random_seeds_hela/epoch=599_val_loss=0.099_rna_splicing_cnn_hela.yaml.ckpt')
model_keras = load_model('custom_adjacency_regularizer_20210731_124_step3.h5')

In [ ]:
model_keras.summary()

In [ ]:
model_keras.get_layer("energy_seq_struct").get_weights()

### Test model on ES7 data, for sanity

In [ ]:
import read_datasets
all_es7_datasets = read_datasets.read_all_datasets()

In [ ]:
LIBRARY_HELA_REP_1 = 11
hela_dataset = all_es7_datasets[LIBRARY_HELA_REP_1]

In [ ]:
hela_dataset["psi"] = hela_dataset.num_exon_inclusion / (hela_dataset.num_exon_inclusion + hela_dataset.num_exon_skipping)

In [ ]:
hela_dataset_sample = hela_dataset.sample(n=10000)

In [ ]:
predictions

In [ ]:
predictions = model_keras(utils.create_input_data([utils.add_flanking(hela_dataset_sample.iloc[i].exon, 10) for i in range(len(hela_dataset_sample))])[:-1]).numpy().flatten()

In [ ]:
pre_tuner_model = Model(inputs=model_keras.inputs, outputs=model_keras.get_layer("energy_seq_struct").output)

In [ ]:
pre_tuner_predictions = pre_tuner_model(utils.create_input_data([utils.add_flanking(hela_dataset_sample.iloc[i].exon, 10) for i in range(len(hela_dataset_sample))])[:-1]).numpy().flatten()

In [ ]:
pre_tuner_predictions

In [ ]:
def sigmoid(x):
    return 1/(1 + np.exp(-x))

In [ ]:
plt.scatter(pre_tuner_predictions, predictions, s=2)
plt.xlabel("Pre-tuner")
plt.ylabel("Final output")


## max gradient seems to be around -1. Compute it and plot.
a, b = sigmoid(np.array(model_keras.get_layer("gen_func")(np.array([[a] for a in [-2.0,0.0]]))).flatten())

x = np.arange(-4,2,0.1)
out = sigmoid(np.array(model_keras.get_layer("gen_func")(np.array([[a] for a in x]))).flatten())
plt.scatter(x,out, s=4)
#plt.scatter(x,x, s=4)
plt.scatter(x, a + (x+2)*(b-a)/2, s=4)

print(f"Max gradient of sigmoid(tuner) = {(b-a)/2:.2f}.")


plt.show()

In [ ]:
def sigmoid_tuner(activations):
    return (sigmoid(np.array(model_keras.get_layer("gen_func")(np.array([[a] for a in activations]))).flatten()))



from sklearn.linear_model import LinearRegression
x = np.arange(-4,0,0.1)
y = np.array(model_keras.get_layer("gen_func")(np.array([[a] for a in x]))).flatten()
lin_reg_model = LinearRegression()
lin_reg_model.fit(x.reshape(-1, 1), y)#.reshape(-1, 1))
print(lin_reg_model.coef_,lin_reg_model.intercept_)

x = np.arange(-4,2,0.1)
y = np.array(model_keras.get_layer("gen_func")(np.array([[a] for a in x]))).flatten()
plt.scatter(x,sigmoid(y))
plt.scatter(x,sigmoid(lin_reg_model.intercept_ + lin_reg_model.coef_*x))
#alpha = model.coef_[0, 0]
#beta = l.predict([[0]])[0, 0]

# Replaces the tuner with a nearby approximation
def convert_final_force_to_logit_scale(x):
    return lin_reg_model.intercept_ + lin_reg_model.coef_*x

In [ ]:
plt.scatter(pre_tuner_predictions, np.log(predictions), s=2)
plt.xlabel("Pre-tuner")
plt.ylabel("Final output (log(PSI))")


## max gradient seems to be around -2.5. Compute it and plot.
a, b = np.log(sigmoid(np.array(model_keras.get_layer("gen_func")(np.array([[a] for a in [-3.0,-2.0]]))).flatten()))

def linear_approximation_of_log_sigmoid_tuner(activation):
    return a + (activation+3)*(b-a)

def sigmoid_tuner(activations):
    return (sigmoid(np.array(model_keras.get_layer("gen_func")(np.array([[a] for a in activations]))).flatten()))

def log_sigmoid_tuner(activations):
    return np.log(sigmoid_tuner(activations))


x = np.arange(-6,3,0.1)
out = np.log(sigmoid(np.array(model_keras.get_layer("gen_func")(np.array([[a] for a in x]))).flatten()))
plt.scatter(x,out, s=4)
plt.scatter(x, linear_approximation_of_log_sigmoid_tuner(x), s=4)

max_gradient_log_sigmoid_tuner = b-a

print(f"Max gradient of log(sigmoid(tuner)) = {max_gradient_log_sigmoid_tuner:.2f}.")

plt.show()

In [ ]:
def sigmoid(z):
    return 1.0/(1.0 + np.exp(-z))

In [ ]:
# Sanity check: model predictions on the HeLa dataset are OK

num_bins = 20
rmse = np.sqrt(((predictions - hela_dataset_sample.psi)**2).mean())
plt.hist2d(predictions, hela_dataset_sample.psi, bins=[num_bins,num_bins], cmap='Blues', norm=matplotlib.colors.LogNorm(vmin=0.6))
plt.xlabel("Prediction")
plt.ylabel("Measurement")
plt.title(f"(rmse={rmse:.2f})")
plt.colorbar()
plt.show()

# Seq: Sequence (90 nt) from -23 to +16 relative to exon (51 nt) ends; sorted by Hexmut#, then position#, then alphabetically. Exon is in red, Variable Hexmut 6mer position is underlined. Position 0 is the relative WT.  See Abbrevition worksheet for explanation of splcing quantification methoids (columns E to  I).

In [ ]:
pre_tuner_model = Model(inputs=model_keras.inputs, outputs=model_keras.get_layer("energy_seq_struct").output)

In [ ]:
ke_et_al_data = pd.read_excel("ke_et_al_Supplemental_Table_S2_no_empty_lines.xls")
ke_et_al_data["PSI"] = 0.19*ke_et_al_data.EI + 0.02
ke_et_al_data["es7_flanked_seq"] = ke_et_al_data.apply(lambda x: utils.add_flanking(x.seq[10:-10],10), axis=1)

In [ ]:
ke_et_al_data["pre_tuner_predictions"] = pre_tuner_model(utils.create_input_data(list(ke_et_al_data.es7_flanked_seq))[:-1]).numpy().flatten()

In [ ]:
ke_et_al_data

In [ ]:
for hex_mut in 'ABCDEFGHIJ':

    filtered_data = ke_et_al_data[ke_et_al_data.Hexmut == hex_mut]

    num_bins = 50
    x = np.log(filtered_data.EI)
    #x = np.log(filtered_data.PSI)
    y = linear_approximation_of_log_sigmoid_tuner(filtered_data.pre_tuner_predictions)
    plt.hist2d(x,y, bins=[num_bins,num_bins], cmap='Blues')#, norm=matplotlib.colors.LogNorm(vmin=0.6))
    plt.xlabel("log(EI)")
    plt.ylabel("Prediction")
    plt.colorbar()

    reg = stats.linregress(x , y)

    left,right = plt.xlim()
    grid = np.arange(left,right,0.01)
    plt.scatter(grid, reg.intercept + grid * reg.slope, s=2, c='r')

    plt.title(f"Slope = {reg.slope:.2f}, intercept = {reg.intercept:.2f}, r = {reg.rvalue:.2f}")

    plt.show()


In [ ]:
for hex_mut in 'ABCDEFGHIJ':

    filtered_data = ke_et_al_data[ke_et_al_data.Hexmut == hex_mut]
    
    wildtype_construct = filtered_data.iloc[0]

    num_bins = 50
    #x = np.log(filtered_data.EI)
    x = (filtered_data.PSI - wildtype_construct.PSI)
    y = 0.28 * (filtered_data.pre_tuner_predictions - wildtype_construct.pre_tuner_predictions)
    plt.hist2d(x,y, bins=[num_bins,num_bins], cmap='Blues')#, norm=matplotlib.colors.LogNorm(vmin=0.6))
    plt.xlabel("Delta PSI")
    plt.ylabel("Max predicted Delta PSI")
    plt.colorbar()

    reg = stats.linregress(x , y)

    left,right = plt.xlim()
    grid = np.linspace(left,right,500)
    plt.scatter(grid, reg.intercept + grid * reg.slope, s=1, c='r')

    plt.title(f"HexMut{hex_mut} WT_PSI = {wildtype_construct.PSI:.2f}, Slope = {reg.slope:.2f}, r = {reg.rvalue:.2f}") # intercept = {reg.intercept:.2f}, 

    plt.show()

In [ ]:
for hex_mut in 'ABCDEFGHIJ':
    filtered_data = ke_et_al_data[ke_et_al_data.Hexmut == hex_mut]
    
    wildtype_construct = filtered_data.iloc[0]

    # find the best shift so the wild type PSI matches the predicted PSI
    all_shifts = np.arange(-5,5,0.1)
    best_shift_index = np.argmin(np.abs(wildtype_construct.PSI - sigmoid_tuner(wildtype_construct.pre_tuner_predictions-np.array(all_shifts))))
    shift = all_shifts[best_shift_index]
    
    num_bins = 50
    #x = np.log(filtered_data.EI)
    x = (filtered_data.PSI)
    y = sigmoid_tuner(filtered_data.pre_tuner_predictions-shift)
    #plt.hist(filtered_data.pre_tuner_predictions)
    #plt.show()
    plt.hist2d(x,y,range=[[0,1],[0,1]], bins=[num_bins,num_bins], cmap='Blues', norm=matplotlib.colors.LogNorm(vmin=0.6))
    plt.xlabel("Measured PSI")
    plt.ylabel("Predicted PSI")
    plt.colorbar()

    reg = stats.linregress(x , y)
    rmse = np.sqrt(((x-y)**2).mean())

    left,right = plt.xlim()
    grid = np.linspace(left,right,500)
    #plt.scatter(grid, reg.intercept + grid * reg.slope, s=1, c='r')
    plt.scatter(grid, grid, s=1, c='r')

    #plt.title(f"HexMut{hex_mut} WT_PSI = {wildtype_construct.PSI:.2f}, shift={shift:.1f}, Slope={reg.slope:.2f}, r={reg.rvalue:.2f}, RMSE={rmse:.2f}") # intercept = {reg.intercept:.2f}, 
    plt.title(f"HexMut{hex_mut} WT_PSI={wildtype_construct.PSI:.2f}, shift={shift:.1f}, r={reg.rvalue:.2f}, RMSE={rmse:.2f}") # intercept = {reg.intercept:.2f}, 

    plt.show()

In [ ]:
plt.hist(ke_et_al_data.PSI)

# Dump data

In [ ]:
to_dump = []

for hex_mut in 'CFHI':
    filtered_data = ke_et_al_data[ke_et_al_data.Hexmut == hex_mut]
    
    wildtype_construct = filtered_data.iloc[0]

    # find the best shift so the wild type PSI matches the predicted PSI
    all_shifts = np.arange(-5,5,0.1)
    best_shift_index = np.argmin(np.abs(wildtype_construct.PSI - sigmoid_tuner(wildtype_construct.pre_tuner_predictions-np.array(all_shifts))))
    shift = all_shifts[best_shift_index]
    
    num_bins = 50
    #x = np.log(filtered_data.EI)
    x = (filtered_data.PSI)
    y = sigmoid_tuner(filtered_data.pre_tuner_predictions-shift)
    
    to_dump.append((x, y))
    
    #plt.hist(filtered_data.pre_tuner_predictions)
    #plt.show()
    plt.hist2d(x,y,range=[[0,1],[0,1]], bins=[num_bins,num_bins], cmap='Blues', norm=matplotlib.colors.LogNorm(vmin=0.6))
    plt.xlabel("Measured PSI")
    plt.ylabel("Predicted PSI")
    plt.colorbar()

    reg = stats.linregress(x , y)
    rmse = np.sqrt(((x-y)**2).mean())

    left,right = plt.xlim()
    grid = np.linspace(left,right,500)
    #plt.scatter(grid, reg.intercept + grid * reg.slope, s=1, c='r')
    plt.scatter(grid, grid, s=1, c='r')

    #plt.title(f"HexMut{hex_mut} WT_PSI = {wildtype_construct.PSI:.2f}, shift={shift:.1f}, Slope={reg.slope:.2f}, r={reg.rvalue:.2f}, RMSE={rmse:.2f}") # intercept = {reg.intercept:.2f}, 
    plt.title(f"HexMut{hex_mut} WT_PSI={wildtype_construct.PSI:.2f}, shift={shift:.1f}, r={reg.rvalue:.2f}, RMSE={rmse:.2f}") # intercept = {reg.intercept:.2f}, 

    plt.show()